In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np


def save_mean(file_name):

    # Collect all evaluation files
    base_dir = Path("")
    files = list(base_dir.glob("stackPredAMR_Results/Evaluation_" + file_name + "_fold*/Evaluation.csv"))

    dfs = [pd.read_csv(f) for f in files]

    # Find common rows
    ab_col = dfs[0].columns[0]
    common_rows = set(dfs[0][ab_col])

    for df in dfs[1:]:
        common_rows &= set(df[ab_col])

    # Print removed rows
    for i, df in enumerate(dfs):
        missing = set(df[ab_col]) - common_rows
        if missing:
            print(f"Removed rows from file index {i}: {sorted(missing)}")

    # Filter and sort all dfs
    filtered_dfs = []
    for df in dfs:
        filtered_df = df[df[ab_col].isin(common_rows)].copy()

        # Ensure same row order in all dfs
        filtered_df = filtered_df.sort_values(by=ab_col).reset_index(drop=True)
        filtered_dfs.append(filtered_df)

    dfs = filtered_dfs

    metric_cols = dfs[0].columns[1:]
    stacked = np.stack([df.iloc[:, 1:].values for df in dfs], axis=0)

    # Compute mean and std
    mean_vals = stacked.mean(axis=0)
    std_vals = stacked.std(axis=0)

    # Build result DataFrames
    mean_df = pd.DataFrame(mean_vals, columns=metric_cols)
    std_df = pd.DataFrame(std_vals, columns=metric_cols)

    # Add AB column
    mean_df.insert(0, ab_col, dfs[0][ab_col])
    std_df.insert(0, ab_col, dfs[0][ab_col])

    # Save
    mean_df.to_csv("Evaluation_mean_" + file_name + ".csv", index=False)
    std_df.to_csv("Evaluation_std_" + file_name + ".csv", index=False)


save_mean("CV_mixedTrain")
save_mean("CV_mixedTrain_test")
save_mean("CV_sensititreTrain")
save_mean("CV_sensititreTrain_test")
save_mean("CV_vitekTrain_test")

Evaluation_CV_mixedTrain_fold*/Evaluation.csv
Evaluation_CV_mixedTrain_test_fold*/Evaluation.csv
Evaluation_CV_sensititreTrain_fold*/Evaluation.csv
Removed rows from file index 0: ['Cefazolin_AB']
Removed rows from file index 1: ['Cefazolin_AB']
Removed rows from file index 3: ['Cefazolin_AB']
Evaluation_CV_sensititreTrain_test_fold*/Evaluation.csv
Removed rows from file index 0: ['Ampicillin-sulbactam_AB', 'Cefazolin_AB', 'Ticarcillin-clavulanic acid_AB']
Removed rows from file index 1: ['Ampicillin-sulbactam_AB', 'Cefazolin_AB', 'Ticarcillin-clavulanic acid_AB']
Removed rows from file index 2: ['Ampicillin-sulbactam_AB', 'Ticarcillin-clavulanic acid_AB']
Evaluation_CV_vitekTrain_test_fold*/Evaluation.csv
